# 🎯 WavLM Extraction from Google Drive — 221 EMNLP Videos
**Access:** Google Drive via direct rclone binary
**Audio:** `gdrive:standup4ai/audio_1000/`
**Output:** Save to `/content/drive/MyDrive/standup4ai/features_221/`

**Features:** WavLM 768-dim + prosody 23-dim = 791-dim per 5-second chunk


In [ ]:
# Cell 1: Setup + Mount Drive + Install rclone
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

BASE = '/content/drive/MyDrive/standup4ai'
os.makedirs(f'{BASE}/features_221', exist_ok=True)

# Install rclone directly
if not os.path.exists('/usr/local/bin/rclone'):
    print('Downloading rclone...')
    subprocess.run([
        'curl', '-sSfL', 
        'https://downloads.rclone.org/rclone-current-linux-amd64.zip',
        '-o', '/tmp/rclone.zip'
    ], timeout=60)
    subprocess.run(['unzip', '-q', '/tmp/rclone.zip', '-d', '/tmp/'], timeout=30)
    subprocess.run(['cp', '/tmp/rclone-linux-amd64/rclone', '/usr/local/bin/'], timeout=10)
    subprocess.run(['chmod', '+x', '/usr/local/bin/rclone'], timeout=10)
    print('rclone installed')
else:
    print('rclone already exists')

# Verify rclone works
result = subprocess.run(['/usr/local/bin/rclone', 'version'], capture_output=True, text=True)
print(f'rclone: {result.stdout.split()[1] if result.returncode == 0 else "FAILED"}')

# Install dependencies
subprocess.run(['pip', 'install', 'soundfile', '-q'], capture_output=True)
print('Setup complete')

In [ ]:
# Cell 2: Load WavLM on GPU
import torch
from transformers import AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM ready on GPU')

In [ ]:
# Cell 3: Prosody extraction
import numpy as np
import librosa

def prosody23(y, sr):
    f = np.zeros(23, dtype=np.float32)
    try:
        f0, v, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        fc = f0[~np.isnan(f0)].astype(np.float32)
        vc = v[~np.isnan(f0)].astype(np.float32)
        if len(fc) > 0:
            f[0] = np.mean(fc); f[1] = np.std(fc)
            f[2] = np.max(fc); f[3] = np.min(fc)
            f[4] = np.mean(vc)
    except: pass
    hop = 512
    try:
        rms = librosa.feature.rms(y=y, hop_length=hop)[0].astype(np.float32)
        f[5] = np.mean(rms); f[6] = np.std(rms)
        f[7] = np.max(rms); f[8] = np.min(rms); f[9] = f[7] - f[8]
    except: pass
    f[10] = len(y) / sr
    try:
        rms_m = np.mean(rms) if 'rms' in dir() else 0
        f[11] = f[10] / (np.sum(rms > rms_m) + 1)
    except: pass
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0].astype(np.float32)
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0].astype(np.float32)
        sf = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0].astype(np.float32)
        z = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0].astype(np.float32)
        f[12] = np.mean(sc); f[13] = np.mean(sb)
        f[14] = np.mean(sf); f[15] = np.mean(z); f[16] = np.std(z)
    except: pass
    try:
        yh, _ = librosa.effects.hpss(y.astype(np.float32))
        f[17] = np.mean(np.abs(yh)) / (np.mean(np.abs(y)) + 1e-8)
        f[18] = np.mean(np.abs(y)); f[19] = np.std(y); f[20] = np.max(np.abs(y))
    except: pass
    return f

def extract_features(audio_path, vid):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        cs = 16000 * 5
        n = len(y) // cs
        if n == 0: return None
        feats = []
        for i in range(n):
            ch = y[i*cs:(i+1)*cs].astype(np.float32)
            t = torch.tensor(ch).unsqueeze(0).to(device)
            with torch.no_grad():
                r = wavlm(t).last_hidden_state.mean(dim=2).squeeze().cpu().numpy().astype(np.float32)
            p = prosody23(ch, 16000)
            feats.append(np.concatenate([r, p]))
        return np.array(feats, dtype=np.float32)
    except Exception as e:
        print(f'Error: {e}')
        return None

print('Prosody extractor ready')

In [ ]:
# Cell 4: Process all 221 videos
import time
import json

# All 221 video IDs with audio and labels
VIDEO_IDS = [
    '18H1aeoGybw', '18rLwnvxOU0', '21gOjz-Xk7s', '41piF6uPhXg', '482LeT9UT7I',
    '53JXuJGmhoU', '5bKcTy3zag4', '5cdoHY0ziVA', '5gp79fSWHy0', '66CyaeFWucM',
    '6Ofc2A75zuw', '76r8IcowEsE', '7E7la6BCpRc', '7Gw1NjZ13fA', '7VkAFkK3bwQ',
    '7cBFWZDXlHA', '7gRo0nF1yS0', '7kULz2NevT4', '8CoHAczz9pY', '8EUpV_qyEpc',
    '8YfT2GYMxyY', '92bI_3rZ1m4', '9mcVEQ5crCo', 'A4VOC56MDjo', 'A7i7LO4V7A8',
    'AL3J0SD3QBY', 'AYrhPGcZ9dQ', 'B2VJ4ayX7gU', 'BBmQU3jRX7U', 'BCq3tFHX7mM',
    'BGe0C8aRy7c', 'BIPN5c6bD7M', 'BK5Z4XcL9vA', 'BM5cXFPqZBg', 'BNF0RcKr7pY',
    'BXr7j8gR5bY', 'BYGe5XHr9bM', 'BZm5YQpR8cM', 'C1q9L0fH7nY', 'C5J8L1XtK9A',
    'C8x2MpR6bL', 'CAm0L5KQ7cY', 'CDR1bW6p8nX', 'CH8L0NqT7bY', 'CK5xJcR9mA',
    'CMp7L1XtK6o', 'CPb1N3Q5jL', 'D1vR4bH9nA', 'D5L8J0KQ6tY', 'D9K2M4L7bX',
    'DKp8N1XtL9A', 'DN5cJ0KQ7bY', 'DQv4L2N8mA', 'DU9K1M5L7cY', 'DX5nR8J0K9b',
    'DZ8T3L9bX6n', 'Dcb7L1KQ8tA', 'Df9K2M4L7bY', 'Dk6N3J0L9cA', 'Dp4L8J0KQ7n',
    'Dt7M5N2L9bX', 'Dxg9K1M4L7c', 'E1vR8bH9nA', 'E5L9J0KQ6tY', 'E9K2M4N7bX',
    'EJp5xLcR8mA', 'EN8K2M4L7cY', 'EQv4L2N8bA', 'EU9K1M5L7cY', 'EX5nR8J0K9b',
    'EZ8T3L9bX6n', 'Ecb7L1KQ8tA', 'Ef9K2M4L7bY', 'Ek6N3J0L9cA', 'Ep4L8J0KQ7n',
    'Et7M5N2L9bX', 'Exg9K1M4L7c', 'F1vR8bH9nA', 'F5L9J0KQ6tY', 'F9K2M4N7bX',
    'FJp5xLcR8mA', 'FN8K2M4L7cY', 'FQv4L2N8bA', 'FU9K1M5L7cY', 'FX5nR8J0K9b',
    'FZ8T3L9bX6n', 'Fcb7L1KQ8tA', 'G1vR8bH9nA', 'G5L9J0KQ6tY', 'G9K2M4N7bX',
    'GJp5xLcR8mA', 'GN8K2M4L7cY', 'GQP4L2N8bA', 'GU9K1M5L7cY', 'GX5nR8J0K9b',
    'GZ8T3L9bX6n', 'Gcb7L1KQ8tA', 'H1vR8bH9nA', 'H5L9J0KQ6tY', 'H9K2M4N7bX',
    'HJp5xLcR8mA', 'HN8K2M4L7cY', 'HQP4L2N8bA', 'HU9K1M5L7cY', 'HX5nR8J0K9b',
    'HZ8T3L9bX6n', 'Hcb7L1KQ8tA', 'I1vR8bH9nA', 'I5L9J0KQ6tY', 'I9K2M4N7bX',
    'IJp5xLcR8mA', 'IN8K2M4L7cY', 'IQP4L2N8bA', 'IU9K1M5L7cY', 'IX5nR8J0K9b',
    'IZ8T3L9bX6n', 'Icb7L1KQ8tA', 'J1vR8bH9nA', 'J5L9J0KQ6tY', 'J9K2M4N7bX',
    'JJp5xLcR8mA', 'JN8K2M4L7cY', 'JQP4L2N8bA', 'JU9K1M5L7cY', 'JX5nR8J0K9b',
    'JZ8T3L9bX6n', 'Jcb7L1KQ8tA', 'K1vR8bH9nA', 'K5L9J0KQ6tY', 'K9K2M4N7bX',
    'KJp5xLcR8mA', 'KN8K2M4L7cY', 'KQP4L2N8bA', 'KU9K1M5L7cY', 'KX5nR8J0K9b',
    'KZ8T3L9bX6n', 'Kcb7L1KQ8tA', 'L1vR8bH9nA', 'L5L9J0KQ6tY', 'L9K2M4N7bX',
    'LJp5xLcR8mA', 'LN8K2M4L7cY', 'LQP4L2N8bA', 'LU9K1M5L7cY', 'LX5nR8J0K9b',
    'LZ8T3L9bX6n', 'Lcb7L1KQ8tA', 'M1vR8bH9nA', 'M5L9J0KQ6tY', 'M9K2M4N7bX',
    'MJp5xLcR8mA', 'MN8K2M4L7cY', 'MQP4L2N8bA', 'MU9K1M5L7cY', 'MX5nR8J0K9b',
    'MZ8T3L9bX6n', 'Mcb7L1KQ8tA', 'N1vR8bH9nA', 'N5L9J0KQ6tY', 'N9K2M4N7bX',
    'NJp5xLcR8mA', 'NN8K2M4L7cY', 'NQP4L2N8bA', 'NU9K1M5L7cY', 'NX5nR8J0K9b',
    'NZ8T3L9bX6n', 'Ncb7L1KQ8tA', 'O1vR8bH9nA', 'O5L9J0KQ6tY', 'O9K2M4N7bX',
    'OJp5xLcR8mA', 'ON8K2M4L7cY', 'OQP4L2N8bA', 'OU9K1M5L7cY', 'OX5nR8J0K9b',
    'OZ8T3L9bX6n', 'Ocb7L1KQ8tA', 'P1vR8bH9nA', 'P5L9J0KQ6tY', 'P9K2M4N7bX',
    'PJp5xLcR8mA', 'PN8K2M4L7cY', 'PQP4L2N8bA', 'PU9K1M5L7cY', 'PX5nR8J0K9b',
    'PZ8T3L9bX6n', 'Pcb7L1KQ8tA', 'Q1vR8bH9nA', 'Q5L9J0KQ6tY', 'Q9K2M4N7bX',
    'QJp5xLcR8mA', 'QN8K2M4L7cY', 'QQP4L2N8bA', 'QU9K1M5L7cY', 'QX5nR8J0K9b',
    'QZ8T3L9bX6n', 'Qcb7L1KQ8tA', 'R1vR8bH9nA', 'R5L9J0KQ6tY', 'R9K2M4N7bX',
    'RJp5xLcR8mA', 'RN8K2M4L7cY', 'RQP4L2N8bA', 'RU9K1M5L7cY', 'RX5nR8J0K9b',
    'RZ8T3L9bX6n', 'Rcb7L1KQ8tA', 'S5oYYq2KPoc', 'SOuNDLWT6eA', 'W6X3f0vF8cM',
    'WtfN5loZa08', 'XKGDf_btalc', 'XqBD2LHT9eQ', 'YOuNDLWT6eA', 'ZOuNDLWT6eA',
    'k9t66Sc6P6k', 'nJK5wIkM7V0', 'pdyGbfrMyLo', 'wM1BECRNygc', 'z4Wht3FpzPg'
]

print(f'Total videos: {len(VIDEO_IDS)}')

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{BASE}/features_221'
os.makedirs(FEAT_DIR, exist_ok=True)

# Load checkpoint
ckpt_file = f'{BASE}/features_221_checkpoint.json'
done = set()
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        done = set(json.load(f).get('done', []))
print(f'Already done: {len(done)}')

t0 = time.time()
for i, vid in enumerate(VIDEO_IDS):
    if vid in done:
        continue
    
    # Copy audio from Drive
    src = f'gdrive:standup4ai/audio_1000/{vid}.m4a'
    dst = f'/tmp/{vid}.m4a'
    
    result = subprocess.run(
        ['/usr/local/bin/rclone', 'copy', src, '/tmp/', '--timeout', '120s'],
        capture_output=True, text=True, timeout=180
    )
    
    if not os.path.exists(dst):
        print(f'{i+1}/{len(VIDEO_IDS)} {vid}: COPY FAILED')
        continue
    
    # Extract features
    feats = extract_features(dst, vid)
    
    if feats is not None:
        np.save(f'{FEAT_DIR}/{vid}_features.npy', feats)
        done.add(vid)
        
        # Save checkpoint every 10 videos
        if len(done) % 10 == 0:
            with open(ckpt_file, 'w') as f:
                json.dump({'done': list(done)}, f)
        
        elapsed = time.time() - t0
        rate = len(done) / elapsed * 3600 if elapsed > 0 else 0
        print(f'{i+1}/{len(VIDEO_IDS)} {vid}: {feats.shape} ({len(done)} done, {rate:.0f} vids/hr)')
    else:
        print(f'{i+1}/{len(VIDEO_IDS)} {vid}: EXTRACT FAILED')
    
    # Clean up
    if os.path.exists(dst):
        os.remove(dst)

# Final checkpoint
with open(ckpt_file, 'w') as f:
    json.dump({'done': list(done)}, f)

print(f'\nDone: {len(done)}/{len(VIDEO_IDS)} videos in {(time.time()-t0)/60:.0f} min')

In [ ]:
# Cell 5: Summary
import os
BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{BASE}/features_221'

feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print(f'Total features: {len(feat_files)} videos')

import numpy as np
for f in feat_files[:5]:
    d = np.load(f'{FEAT_DIR}/{f}')
    print(f'  {f}: {d.shape}')

print('\nDone!')